In [25]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import sys
import os
import torch
import torchvision
import copy

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [27]:
from src.trainer.BufferTrainer import BufferTrainer
from src.data_utils import (
    get_mnist_tasks,
    _extract_targets,
    get_context_sets,
    create_holdout_set,
)
from src.utils.general import InContextHead, print_colored, set_seed
from src import models
from src.buffer import MultiTaskBuffer
from src.data_utils import split_mnist_by_labels, get_context_sets
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

from configs import MNIST_BUFFER_CONFIG as CONFIG

In [28]:
device = "cuda"
classes = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

task_pairs = [
    ("cat", "truck"),
    ("frog", "ship"),
    ("horse", "automobile"),
    ("dog", "airplane"),
    ("bird", "deer"),
]
task_pairs_ids = [(classes.index(i), classes.index(j)) for i, j in task_pairs]
animals_mask = torch.tensor([0, 0, 1, 1, 1, 1, 1, 1, 0, 0]).to(device)

transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize((224, 224)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3),
    ]
)


def domain_map_fn(y):
    return animals_mask[y]


@torch.no_grad()
def encode(x, model, device="cuda"):
    # Handle batching to avoid out-of-memory issues
    batch_size = 2048
    features = []
    for i in range(0, len(x), batch_size):
        batch = x[i : i + batch_size].to(device)
        batch_features = model(batch).flatten(start_dim=1).cpu()
        features.append(batch_features)
    return torch.cat(features, dim=0)


def encode_dataset(train_dataset, test_dataset, encoder, device="cuda"):
    train_imgs, train_labels = train_dataset.data, train_dataset.targets
    test_imgs, test_labels = test_dataset.data, test_dataset.targets
    # apply the appropriate scaling and transposition
    train_imgs = (
        torch.tensor(train_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    )
    test_imgs = torch.tensor(test_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    train_imgs = (train_imgs - 0.5) / 0.5
    test_imgs = (test_imgs - 0.5) / 0.5
    train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
    test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()

    if encoder is not None:
        train_imgs = encode(train_imgs, encoder, device)
        test_imgs = encode(test_imgs, encoder, device)

    train_dataset = torch.utils.data.TensorDataset(train_imgs, train_labels)
    test_dataset = torch.utils.data.TensorDataset(test_imgs, test_labels)
    return train_dataset, test_dataset


def get_tasks(encoder):
    non_animals = [0, 1, 8, 9]
    animals = [2, 3, 4, 5, 6, 7]

    non_animal_indices = torch.tensor(non_animals)[torch.randperm(4)].tensor_split(5)
    animal_indices = list(torch.tensor(animals)[torch.randperm(6)].tensor_split(5))
    animal_indices.reverse()

    task_pairs_ids = [
        t.tolist() + n.tolist() for t, n in zip(animal_indices, non_animal_indices)
    ]
    print("Contexts:", task_pairs_ids)
    trainset = torchvision.datasets.CIFAR10(
        root="./data", train=True, download=True, transform=transform
    )
    testset = torchvision.datasets.CIFAR10(
        root="./data", train=False, download=True, transform=transform
    )
    trainset.targets = torch.tensor(trainset.targets)
    testset.targets = torch.tensor(testset.targets)

    if encoder is not None:
        trainset, testset = encode_dataset(trainset, testset, encoder)
    test_tasks = [
        split_mnist_by_labels(testset, task_pair_id) for task_pair_id in task_pairs_ids
    ]
    train_tasks = [
        split_mnist_by_labels(trainset, task_pair_id) for task_pair_id in task_pairs_ids
    ]

    return train_tasks, test_tasks

In [29]:
# Get the CIFAR 100 dataset
cifar100_trainset = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=True, transform=transform
)
cifar100_testset = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=transform
)

# Convert targets to tensor for consistency
cifar100_trainset.targets = torch.tensor(cifar100_trainset.targets)
cifar100_testset.targets = torch.tensor(cifar100_testset.targets)

# Print dataset info
print(f"CIFAR-100 training set: {len(cifar100_trainset)} images")
print(f"CIFAR-100 test set: {len(cifar100_testset)} images")
print(f"Number of classes: {len(set(cifar100_trainset.targets.tolist()))}")

# Sample a few class names
print(f"Sample classes: {cifar100_trainset.classes[:10]}")

CIFAR-100 training set: 50000 images
CIFAR-100 test set: 10000 images
Number of classes: 100
Sample classes: ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle']


In [30]:
# Create a simple function to load the model
save_dir = os.path.join(project_root, "notebooks/saved_models")
model_path = os.path.join(save_dir, "resnet18_cifar100_pretrained.pth")


def load_pretrained_model(model_path, num_classes=100):
    model = torchvision.models.resnet18(weights=None)
    model.fc = torch.nn.Linear(512, num_classes)
    model.load_state_dict(torch.load(model_path))
    return model


model = load_pretrained_model(model_path)

In [31]:
encoder = copy.deepcopy(model).cuda()
encoder.fc = torch.nn.Identity()

In [32]:
X_min, X_max = None, None
for task in get_tasks(encoder):
    X, _ = next(
        iter(torch.utils.data.DataLoader(task[0], batch_size=10000, shuffle=False))
    )
    if X_min is None:
        X_min, X_max = X.min(dim=0).values, X.max(dim=0).values
    else:
        X_min = torch.min(X_min, X.min(dim=0).values)
        X_max = torch.max(X_max, X.max(dim=0).values)
X_min, X_max = X_min.to(device), X_max.to(device)

Contexts: [[7, 8], [3, 9], [5, 1], [6, 0], [2, 4]]


/tmp/ipykernel_1425338/3354533864.py:60: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_1425338/3354533864.py:61: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()


In [33]:
import wandb

from configs import CIFAR_BUFFER_CONFIG as CONFIG
print(CONFIG.keys())

dict_keys(['projection_strategy', 'n_certificate_samples', 'min_acc_limit', 'min_acc_increment', 'n_iters', 'primal_learning_rate', 'dual_learning_rate', 'penalty_coefficient', 'checkpoint', 'l2_lambda', 'unbias_lambda', 'lr', 'weight_decay', 'epochs', 'batch_size', 'loosening_thresh', 'loosening_step', 'buffer_k'])


In [ ]:
SMALL = 1000
MEDIUM = 5000
LARGE = 15000


def run_buffer(buffer_size: int, seed: int, config: wandb.config, paradigm="TIL"):
    """
    ---------------------------------------------
    REQUIRED STUFF
    ---------------------------------------------
    """

    device = "cuda"
    classes = [
        "airplane",
        "automobile",
        "bird",
        "cat",
        "deer",
        "dog",
        "frog",
        "horse",
        "ship",
        "truck",
    ]

    task_pairs = [
        ("cat", "truck"),
        ("frog", "ship"),
        ("horse", "automobile"),
        ("dog", "airplane"),
        ("bird", "deer"),
    ]
    task_pairs_ids = [(classes.index(i), classes.index(j)) for i, j in task_pairs]
    animals_mask = torch.tensor([0, 0, 1, 1, 1, 1, 1, 1, 0, 0]).to(device)

    transform = torchvision.transforms.Compose(
        [
            torchvision.transforms.Resize((224, 224)),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3),
        ]
    )

    def domain_map_fn(y):
        return animals_mask[y]

    @torch.no_grad()
    def encode(x, model, device="cuda"):
        # Handle batching to avoid out-of-memory issues
        batch_size = 2048
        features = []
        for i in range(0, len(x), batch_size):
            batch = x[i : i + batch_size].to(device)
            batch_features = model(batch).flatten(start_dim=1).cpu()
            features.append(batch_features)
        return torch.cat(features, dim=0)

    def encode_dataset(train_dataset, test_dataset, encoder, device="cuda"):
        train_imgs, train_labels = train_dataset.data, train_dataset.targets
        test_imgs, test_labels = test_dataset.data, test_dataset.targets
        # apply the appropriate scaling and transposition
        train_imgs = (
            torch.tensor(train_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
        )
        test_imgs = (
            torch.tensor(test_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
        )
        train_imgs = (train_imgs - 0.5) / 0.5
        test_imgs = (test_imgs - 0.5) / 0.5
        train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
        test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()

        if encoder is not None:
            train_imgs = encode(train_imgs, encoder, device)
            test_imgs = encode(test_imgs, encoder, device)

        train_dataset = torch.utils.data.TensorDataset(train_imgs, train_labels)
        test_dataset = torch.utils.data.TensorDataset(test_imgs, test_labels)
        return train_dataset, test_dataset

    def get_tasks(encoder, seed=42):
        set_seed(seed)
        non_animals = [0, 1, 8, 9]
        animals = [2, 3, 4, 5, 6, 7]

        non_animal_indices = torch.tensor(non_animals)[torch.randperm(4)].tensor_split(
            5
        )
        animal_indices = list(torch.tensor(animals)[torch.randperm(6)].tensor_split(5))
        animal_indices.reverse()

        task_pairs_ids = [
            t.tolist() + n.tolist() for t, n in zip(animal_indices, non_animal_indices)
        ]
        print("Contexts:", task_pairs_ids)
        trainset = torchvision.datasets.CIFAR10(
            root="./data", train=True, download=True, transform=transform
        )
        testset = torchvision.datasets.CIFAR10(
            root="./data", train=False, download=True, transform=transform
        )
        trainset.targets = torch.tensor(trainset.targets)
        testset.targets = torch.tensor(testset.targets)

        if encoder is not None:
            trainset, testset = encode_dataset(trainset, testset, encoder)
        test_tasks = [
            split_mnist_by_labels(testset, task_pair_id)
            for task_pair_id in task_pairs_ids
        ]
        train_tasks = [
            split_mnist_by_labels(trainset, task_pair_id)
            for task_pair_id in task_pairs_ids
        ]

        return train_tasks, test_tasks

    # Sample a few class names
    print(f"Sample classes: {cifar100_trainset.classes[:10]}")

    # Create a simple function to load the model
    def load_pretrained_model(model_path, num_classes=100):
        model = torchvision.models.resnet18(weights=None)
        model.fc = torch.nn.Linear(512, num_classes)
        model.load_state_dict(torch.load(model_path))
        return model

    model = load_pretrained_model(model_path)
    encoder = copy.deepcopy(model).cuda()
    encoder.fc = torch.nn.Identity()

    X_min, X_max = None, None
    for task in get_tasks(encoder, seed):
        X, _ = next(
            iter(torch.utils.data.DataLoader(task[0], batch_size=10000, shuffle=False))
        )
        if X_min is None:
            X_min, X_max = X.min(dim=0).values, X.max(dim=0).values
        else:
            X_min = torch.min(X_min, X.min(dim=0).values)
            X_max = torch.max(X_max, X.max(dim=0).values)
    X_min, X_max = X_min.to(device), X_max.to(device)

    """
    ---------------------------------------------
    REQUIRED STUFF
    ---------------------------------------------
    """

    SEED = seed
    CONFIG = config
    set_seed(SEED)
    train_tasks, test_tasks = get_tasks(encoder, SEED)

    context_sets = get_context_sets(test_tasks)
    head = InContextHead(context_sets, 10, device="cuda")
    head.set_context(0)
    model = models.get_fully_connected_model(
        input_dim=512,
        seed=seed,
        device="cuda",
        output_dim=2 if paradigm == "DIL" else 10,
        head=head if paradigm == "TIL" else None,
    )
    print(
        f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
    )

    unbias = UnbiasRegulariser(
        lmbd=CONFIG["unbias_lambda"],
        unbias_domain=[X_min, X_max],
    )
    l2 = L2Regulariser(lmbd=CONFIG["l2_lambda"])
    regulariser = MultiRegulariser([l2, unbias])

    if buffer_size == SMALL:
        sizes = [400, 200, 200, 200, 0]
    elif buffer_size == MEDIUM:
        sizes = [1400, 1200, 800, 600, 0]
    elif buffer_size == LARGE:
        sizes = [4800, 4000, 4000, 3200, 0]
    train_tasks, buffer_tasks = zip(
        *[
            create_holdout_set(dataset, holdout_size=holdout)
            for dataset, holdout in zip(train_tasks, sizes)
        ]
    )
    print([len(task) for task in buffer_tasks])
    print([len(task) for task in train_tasks])

    task_labels = [
        torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks
    ]
    buffer_labels = [
        torch._unique(_extract_targets(train))[0].tolist() for train in buffer_tasks
    ]
    print(task_labels)
    print(buffer_labels)

    buffer = MultiTaskBuffer([])
    buffer_trainer = BufferTrainer(
        model,
        checkpoint=CONFIG["checkpoint"],
        n_iters=CONFIG["n_iters"],
        min_acc_limit=CONFIG["min_acc_limit"],
        min_acc_increment=0,
        primal_learning_rate=CONFIG["primal_learning_rate"],
        dual_learning_rate=CONFIG["dual_learning_rate"],
        projection_strategy=CONFIG["projection_strategy"],
        n_certificate_samples=CONFIG["n_certificate_samples"],
        penalty_coefficient=CONFIG["penalty_coefficient"],
        paradigm=paradigm,
        seed=SEED,
        buffer=buffer,
        domain_map_fn=domain_map_fn if paradigm == "DIL" else None,
        task_labels=task_labels,
    )

    if buffer_size == SMALL:
        MAX_BUFFER_CALLS = 1
    if buffer_size == MEDIUM:
        MAX_BUFFER_CALLS = 3
    if buffer_size == LARGE:
        MAX_BUFFER_CALLS = 7

    targets = {
        "TIL": 0.65,
        "DIL": 0.75,
        "CIL": 0.55
    }
    target_acc = targets[paradigm]
    lower_bounds = []
    buffer_calls = []
    accuracy_matrix = []
    for i, (train, test, buffer) in enumerate(
        zip(train_tasks, test_tasks, buffer_tasks)
    ):
        buffer_trainer.train(
            train,
            test,
            batch_size=CONFIG["batch_size"],
            epochs=CONFIG["epochs"],
            lr=CONFIG["lr"],
            weight_decay=CONFIG["weight_decay"],
            regulariser=regulariser,
            context_id=i if paradigm == "TIL" else None,
        )
        results = buffer_trainer.test(
            test_tasks,
            context_list=list(range(len(test_tasks)))
            if paradigm == "TIL"
            else [None] * len(test_tasks),
        )
        accs = [res[1] for res in results]
        if i == 0 and accs[0] < 0.65:
            wandb.finish(1)
            return
        # If results are not satisfactory, then use buffer data to recompute rashomon set and continue training
        j = 0
        buffer_call = 0
        prev_acc = None
        while (
            j < MAX_BUFFER_CALLS
            and results[i][1] < target_acc
            and i > 0
            and not buffer_trainer.buffer.is_empty()
        ):
            buffer_call += 1
            print_colored("Using buffer to recompute LID.", color="amber")

            buffer_trainer.recall_dataset, (buffer_X, buffer_y) = (
                buffer_trainer.buffer.consume_merge()
            )
            print("Recall dataset size:", len(buffer_trainer.recall_dataset))
            dataset = torch.utils.data.TensorDataset(buffer_X, buffer_y)
            buffer_trainer.test(train_tasks, context_list=list(range(5)) if paradigm=="TIL" else [None] * 5)
            buffer_trainer.compute_rashomon_set(
                dataset,
                use_outer_bbox=False,
                batch_size=len(dataset),
                context_id=i-1 if paradigm == "TIL" else None,
            )
            buffer_trainer.train(
                train,
                test,
                batch_size=CONFIG["batch_size"],
                epochs=CONFIG["epochs"],
                lr=CONFIG["lr"],
                weight_decay=CONFIG["weight_decay"],
                regulariser=regulariser,
                early_stopping=True,
                val_freq=25,
                patience=5,
                context_id=i if paradigm == "TIL" else None,
            )
            results = buffer_trainer.test(
                test_tasks,
                context_list=list(range(len(test_tasks)))
                if paradigm == "TIL"
                else [None] * len(test_tasks),
            )
            accs = [res[1] for res in results]

            print("lower_bounds:", lower_bounds)
            diffs = [accs[i] - lower_bounds[i] for i in range(len(lower_bounds))]
            min_diff_idx = diffs.index(
                min(diffs)
            )  # The assumption is that the task closest to its boundary is the one restricting further improvements
            if results[i][1] > target_acc:
                break
            elif (
                prev_acc is not None
                and results[i][1] - prev_acc < CONFIG["loosening_thresh"]
            ):
                print("Loosening task", min_diff_idx, "bounds.")
                lower_bounds[min_diff_idx] = max(
                    lower_bounds[min_diff_idx] - CONFIG["loosening_step"], 0.001
                )
                buffer_trainer.min_acc_limit = lower_bounds
            prev_acc = results[i][1]
            j += 1
        buffer_calls.append(buffer_call)

        print_colored(accs, color="green")
        accuracy_matrix.append(accs)

        lower_bounds.append(max(accs[i] - CONFIG["min_acc_increment"], 0.001))

        buffer_trainer.min_acc_limit = lower_bounds

        if buffer_trainer._last_projection is not None:
            buffer_trainer.final_certificates.append(
                buffer_trainer.certificates[buffer_trainer._last_projection]
            )
        if i < len(train_tasks) - 1:
            buffer_trainer.compute_rashomon_set(
                test, context_id=i if paradigm == "TIL" else None
            )
            if len(buffer):
                buffer_trainer.add_to_buffer(buffer, task_id=i, k=200)
        else:
            print("Buffer calls:", buffer_calls)
            print("lower_bounds:", lower_bounds)
            accuracy_matrix.append(lower_bounds)
            columns = [f"Test Task {i}" for i in range(len(test_tasks))]
            rows = [f"Task {i}" for i in range(len(test_tasks))] + ["Certificates"]
            wandb.log(
                {
                    "accuracy_matrix": wandb.Table(
                        data=accuracy_matrix, columns=columns, rows=rows
                    )
                }
            )

    wandb.finish(0)


# for paradigm in ["DIL", "CIL", "TIL"]:
for paradigm in ["TIL"]:
    for buffer_label, buffer_size in [
        ("large", LARGE),
        ("medium", MEDIUM),
        ("small", SMALL),
    ]:
        for i in range(5, 15):
            with wandb.init(
                project="certified-continual-learning",
                config=CONFIG,
                reinit=True,
                tags=[
                    "final_cifar_buffer",
                    f"buffer_{buffer_label}",
                    f"buffer_{paradigm.lower()}",
                ]
            ):
                set_seed(i)
                wandb.log({"seed": i})
                config = wandb.config
                run_buffer(buffer_size, i, config, paradigm=paradigm)

Sample classes: ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle']
Contexts: [[2, 9], [7, 1], [3, 0], [6, 8], [5, 4]]


/tmp/ipykernel_1425338/3402911945.py:71: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_1425338/3402911945.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()


Contexts: [[2, 9], [7, 1], [3, 0], [6, 8], [5, 4]]
Tasks: [[2, 9], [1, 7], [0, 3], [6, 8], [4, 5]]
[4800, 4000, 4000, 3200, 0]
[5200, 6000, 6000, 6800, 10000]
[[2, 9], [1, 7], [0, 3], [6, 8], [4, 5]]
[[2, 9], [1, 7], [0, 3], [6, 8], []]


Training Epochs: 100%|███████████████████████████████████████████████| 5/5 [00:01<00:00,  4.87it/s, val_loss=0.0000, val_acc=None, proj=None, progress=0.98]


Test Results: [(0.3615, 0.8695), (2.3026, 0.431), (2.3026, 0.3095), (2.2982, 0.5), (2.2988, 0.5)] (Avg: (1.9127, 0.5220))
[0.8695, 0.431, 0.3095, 0.5, 0.5]
Initial acc constraint violation: -0.1960 (Positive = violated)
[0.6695, 0.0, 0.0, 0.0, 0.0]
Number of model parameters: 5130
Computing Rashomon set with min acc limit: 0.67
Initial bbox:  Obj=16.00,  Size=82080.00,  Min acc hard=0.86,  Min acc soft=0.87


...

100%|██████████████████████████████████████████████████████████████████████| 200/200 [00:02<00:00, 71.61it/s, size=61500.59, obj=11.988, min_soft_acc=0.635]


Final bbox:  Obj=11.99,  Size=61500.59,  Min acc hard=0.64,  Min acc soft=0.64
Computing final certificates over 400 samples
Checkpointed every 2 iterations for a total of 100 checkpoints
Checkpoints sizes: ['61493.23', '61493.56', '61493.97', '61494.46', '61495.05', '61495.77', '61496.64', '61497.70', '61498.48', '61498.00', '61497.47', '61496.94', '61496.52', '61496.22', '61496.10', '61496.18', '61496.50', '61497.14', '61498.06', '61498.61', '61497.96', '61497.81', '61498.00', '61498.46', '61498.80', '61499.06', '61499.26', '61498.39', '61498.02', '61498.00', '61498.20', '61498.59', '61499.23', '61499.86', '61499.38', '61498.39', '61497.82', '61497.57', '61497.55', '61497.70', '61498.04', '61498.57', '61499.28', '61499.46', '61499.35', '61499.37', '61499.27', '61499.41', '61499.75', '61499.84', '61499.06', '61498.71', '61498.62', '61498.72', '61499.00', '61499.50', '61500.04', '61499.84', '61499.61', '61499.63', '61499.62', '61499.82', '61499.96', '61500.05', '61499.92', '61499.32', 

Training Epochs:  80%|████████████████████████████████████████          | 4/5 [00:09<00:02,  2.37s/it, val_loss=0.0000, val_acc=None, proj=0, progress=0.87]
Traceback (most recent call last):
  File "/tmp/ipykernel_1425338/3402911945.py", line 380, in <module>
    run_buffer(buffer_size, i, config, paradigm=paradigm)
  File "/tmp/ipykernel_1425338/3402911945.py", line 239, in run_buffer
    buffer_trainer.train(
  File "/data/le24/CertifiedContinualLearning/src/trainer/BufferTrainer.py", line 71, in train
    return super().train(
  File "/data/le24/CertifiedContinualLearning/src/trainer/BaseTrainer.py", line 122, in train
    self.model = self._train(
  File "/data/le24/CertifiedContinualLearning/src/trainer/IntervalTrainer.py", line 254, in _train
    loss, _ = self._train_step(
  File "/data/le24/CertifiedContinualLearning/src/trainer/BufferTrainer.py", line 104, in _train_step
    return super()._train_step(
  File "/data/le24/CertifiedContinualLearning/src/trainer/IntervalTrainer.

seed,▁
seed,5


KeyboardInterrupt: 